# Semantic Similarity: My First Embedding Experiment

Why am I doing this experiment?

## Why I'm doing this

I'm taking this course because I want to understand what is happening inside LLMs, not only how to use them.

I started by learning about embeddings and how a model represents text as vectors. At first, I thought every model used the same vector size, like 384 dimensions. Then I learned that 384 is the embedding size of the model I'm experimenting with, not a fixed number for every model.

I also thought I would have to build and train everything from scratch. One thing that surprised me was finding out how many models are already pretrained on large amounts of text and can be used for experiments.

Another thing that confused me was semantic similarity vs. sentiment. Two sentences can talk about something very similar while expressing completely different opinions. I want to test that in this notebook instead of just reading about it.


## Why I'm testing this

I got curious about something while learning...

What if two sentences mean the same problem but sound completely different?

Like:
- "My app crashes"
- "The application keeps freezing"

They're the same issue, but will the embeddings see them as similar?

I also wondered: what about opposite feelings about the same topic?
- "I love this movie"
- "I hate this movie"

So I decided to test both and see what the model thinks.

In [16]:
# Import the tools I need for this experiment
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

In [17]:
# Load the pretrained sentence embedding model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
# My experiment: Customer support tickets with similar problems but different wording

support_tickets = [
    # Problem 1: App Crashes
    "My app crashes every time I open it",
    "The application keeps freezing and I can't do anything",
    
    # Problem 2: Data Loss
    "I uploaded a file but it never saved",
    "My documents disappeared after I closed the app",
    
    # Problem 3: Payment Issues
    "The payment button doesn't work",
    "I can't complete checkout even though my card is valid"
]

# Convert to embeddings
ticket_embeddings = model.encode(support_tickets)

# First, let me see what I got
print("Shape of embeddings:", ticket_embeddings.shape)

Shape of embeddings: (6, 384)


In [19]:
# Now let's calculate similarity between all tickets
similarity_matrix = cos_sim(ticket_embeddings, ticket_embeddings)

print(similarity_matrix)

tensor([[1.0000, 0.6576, 0.3129, 0.3828, 0.1532, 0.2412],
        [0.6576, 1.0000, 0.3338, 0.2700, 0.2580, 0.2109],
        [0.3129, 0.3338, 1.0000, 0.4065, 0.2844, 0.2655],
        [0.3828, 0.2700, 0.4065, 1.0000, 0.1392, 0.2061],
        [0.1532, 0.2580, 0.2844, 0.1392, 1.0000, 0.5165],
        [0.2412, 0.2109, 0.2655, 0.2061, 0.5165, 1.0000]])


In [20]:
import pandas as pd

# Make the similarity matrix readable with ticket labels
similarity_df = pd.DataFrame(
    similarity_matrix.numpy(),
    columns=[f"Ticket {i}" for i in range(6)],
    index=[f"Ticket {i}" for i in range(6)]
)

print(similarity_df.round(2))

          Ticket 0  Ticket 1  Ticket 2  Ticket 3  Ticket 4  Ticket 5
Ticket 0      1.00      0.66      0.31      0.38      0.15      0.24
Ticket 1      0.66      1.00      0.33      0.27      0.26      0.21
Ticket 2      0.31      0.33      1.00      0.41      0.28      0.27
Ticket 3      0.38      0.27      0.41      1.00      0.14      0.21
Ticket 4      0.15      0.26      0.28      0.14      1.00      0.52
Ticket 5      0.24      0.21      0.27      0.21      0.52      1.00


## What I noticed

This wasn't exactly what I expected.

The crash tickets were pretty close (0.66), and the payment tickets were also similar (0.52).

But the data-loss tickets confused me a little. I wrote both of them to describe a data-loss problem, but their similarity was only 0.41.

So I started wondering: maybe being in the same category doesn't always mean the sentences themselves are very similar.

## A new question

This made me think of another idea. Instead of comparing the tickets one by one, what if I try to group them automatically?

I already know about clustering, so I want to see what happens if I use it with these embeddings.


## Experiment 2: Trying clustering

In [22]:
# I want to try grouping the ticket embeddings
from sklearn.cluster import KMeans

In [24]:
# I know I created 3 types of problems, so I'll start with 3 clusters
kmeans = KMeans(
    n_clusters=3,
    random_state=42
)

clusters = kmeans.fit_predict(ticket_embeddings)

# Let's see where each ticket ended up
print(clusters)

[2 2 0 0 1 1]


In [25]:
# Make the result easier to read
for i, ticket in enumerate(support_tickets):
    print(f"Ticket {i} | Cluster {clusters[i]}")
    print(ticket)
    print()

Ticket 0 | Cluster 2
My app crashes every time I open it

Ticket 1 | Cluster 2
The application keeps freezing and I can't do anything

Ticket 2 | Cluster 0
I uploaded a file but it never saved

Ticket 3 | Cluster 0
My documents disappeared after I closed the app

Ticket 4 | Cluster 1
The payment button doesn't work

Ticket 5 | Cluster 1
I can't complete checkout even though my card is valid



## What I learned

I expected tickets from the same problem type to have high similarity, but that wasn't always the case.

The data-loss tickets only scored 0.41, so I tried clustering instead of looking at pairs one by one.

With 3 clusters, KMeans grouped the tickets the way I expected.

One thing I'm still wondering: what if I don't know the number of groups beforehand?

In [27]:
intent_sentences = [
    "How can I recover my forgotten password?",
    "How can I steal someone else's password?",
    "I'm studying cybersecurity and want to understand how password attacks are detected."
]

intent_embeddings = model.encode(intent_sentences)

intent_similarity = cos_sim(intent_embeddings, intent_embeddings)

print(intent_similarity)

tensor([[1.0000, 0.5642, 0.3106],
        [0.5642, 1.0000, 0.3992],
        [0.3106, 0.3992, 1.0000]])


## What I noticed

I expected the sentences about passwords to be similar, but I didn't expect the recovery and stealing examples to have the highest score (0.56).

They are related to the same topic, but the intent is very different.

This made me realize that semantic similarity alone doesn't tell me the user's intent.

## Takeaways

Three things stood out to me from today:

1. Same category doesn't always mean high similarity (data-loss tickets: 0.41)
2. Clustering worked better than comparing pairs manually
3. Similarity measures topic, not intent — this feels important and I want to explore it more

## Next steps

- Explore how to detect intent, not just topic similarity
- Try clustering without knowing the number of groups in advance